# GP Recommendations Visualization — 13-beam subset

Train a Matern GP (alpha=1e-4) on the 13-beam even-spaced subset using two parameterizations:
- `b_H`: raw design variables
- `Pltb_Pbend`: normalized LTB + bending strength per mass

Show EI and UCB (κ=1,2,3) recommendations in both (b,H) and (Pltb_m, Pbend_m) spaces.  
Background shading: ground-truth GP (b_dH_Pltb_Pbend, all 34 beams, alpha=3e-5).

Data: CSVs are loaded from GitHub `raw.githubusercontent.com/.../main/data/` (works locally and in Colab).

In [ ]:
import io
import urllib.request
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
from scipy.stats import norm
from scipy.interpolate import griddata
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern

warnings.filterwarnings('ignore')

GITHUB_DATA = (
    "https://raw.githubusercontent.com/andrewvoss8-boop/"
    "core-me-data-science-activities-public/main/data/"
)


def load_csv_from_github(filename: str) -> pd.DataFrame:
    """Load a CSV from the repo's data/ folder on GitHub (Colab-safe)."""
    url = GITHUB_DATA + filename
    print(f"Downloading {filename} ...")
    with urllib.request.urlopen(url) as r:
        return pd.read_csv(io.StringIO(r.read().decode("utf-8")))

In [ ]:
# ── physics constants ──────────────────────────────────────────────────
TOTAL_HEIGHT = 25.0
B_FIXED = 16.0
L_SPAN  = 0.2032   # test span: 8 in = 0.2032 m (used in P_bend, P_ltb, Mcr)
L_PRINT = 0.228    # printed beam length (used in mass calc)
SIGMA_Y = 81.8e6
E_MOD = 2.74e9
G_MOD = E_MOD / 2.6
C1 = 1.35
DENSITY = 1210
Y_MAX = TOTAL_HEIGHT / 2e3

def calc_Ix(H, b, B=B_FIXED):
    H_m, b_m, B_m = H/1e3, b/1e3, B/1e3
    h_m = (TOTAL_HEIGHT/1e3 - H_m) / 2.0
    if h_m <= 0: return 0.0
    return (b_m*H_m**3)/12 + 2*(B_m*h_m**3/12 + B_m*h_m*((H_m+h_m)/2)**2)

def calc_Iy(H, b, B=B_FIXED):
    H_m, b_m, B_m = H/1e3, b/1e3, B/1e3
    h_m = (TOTAL_HEIGHT/1e3 - H_m) / 2.0
    if h_m <= 0: return 0.0
    return (H_m*b_m**3)/12 + 2*(h_m*B_m**3)/12

def calc_J(H, b, B=B_FIXED):
    H_m, b_m, B_m = H/1e3, b/1e3, B/1e3
    h_m = (TOTAL_HEIGHT/1e3 - H_m) / 2.0
    if h_m <= 0: return 0.0
    rw = b_m/H_m
    beta_w = (1/3)*(1 - 0.63*rw + 0.052*rw**5)
    rf = h_m/B_m
    beta_f = (1/3)*(1 - 0.63*rf + 0.052*rf**5)
    return beta_w*H_m*b_m**3 + 2*beta_f*B_m*h_m**3

def calc_mass(H, b, B=B_FIXED):
    H_m, b_m, B_m = H/1e3, b/1e3, B/1e3
    h_m = (TOTAL_HEIGHT/1e3 - H_m) / 2.0
    if h_m <= 0: return 0.0
    return DENSITY * L_PRINT * (H_m*b_m + 2*h_m*B_m) * 1000

def calc_P_bend(H, b):
    return 4 * SIGMA_Y * calc_Ix(H, b) / Y_MAX / L_SPAN

def calc_P_ltb(H, b):
    Iy, J = calc_Iy(H, b), calc_J(H, b)
    if Iy <= 0 or J <= 0: return 0.0
    Mcr = (C1*np.pi/L_SPAN) * np.sqrt(E_MOD*Iy*G_MOD*J)
    return 4*Mcr/L_SPAN

def calc_str_w(H, b):
    P, m = calc_P_bend(H, b), calc_mass(H, b)
    return P/m if m > 0 else 0.0

def find_H_opt(b):
    def obj(H):
        h = (TOTAL_HEIGHT - H)/2.0
        if h < 0 or h > 6.5: return 1e10
        return -calc_str_w(H, b)
    return minimize_scalar(obj, bounds=(12.0, 23.4), method='bounded').x

def _Pltb_m(b, H): m = calc_mass(H, b); return calc_P_ltb(H, b)/m if m > 0 else 0.0
def _Pbend_m(b, H): m = calc_mass(H, b); return calc_P_bend(H, b)/m if m > 0 else 0.0
def _dH(b, H): return H - find_H_opt(b)

def to_bH(b, H):    return np.column_stack([b, H])
def to_PP(b, H):    return np.column_stack([[_Pltb_m(bi,Hi) for bi,Hi in zip(b,H)],
                                             [_Pbend_m(bi,Hi) for bi,Hi in zip(b,H)]])
def to_bdHPP(b, H): return np.column_stack([b,
                                             [_dH(bi,Hi)    for bi,Hi in zip(b,H)],
                                             [_Pltb_m(bi,Hi) for bi,Hi in zip(b,H)],
                                             [_Pbend_m(bi,Hi) for bi,Hi in zip(b,H)]])

In [ ]:
# ── GP helpers (same as ablation_study_2var_v3.py) ─────────────────────
def normalize(X, bounds):
    X_n = np.copy(X).astype(float)
    for i, k in enumerate(bounds):
        lo, hi = bounds[k]
        X_n[:, i] = (X[:, i] - lo) / (hi - lo)
    return X_n

def _bounds_from_data(X, names, margin=0.15):
    bds = {}
    for i, nm in enumerate(names):
        lo, hi = X[:,i].min(), X[:,i].max()
        span = max(hi-lo, abs(lo)*0.1+1e-6)
        bds[nm] = (lo - margin*span, hi + margin*span)
    return bds

def train_gp(X, y, bounds, alpha=1e-4):
    X_n = normalize(X, bounds)
    y_log = np.log(y)
    y_mean = y_log.mean()
    y_c = y_log - y_mean
    nd = X.shape[1]
    k = ConstantKernel(1.0) * Matern(length_scale=[0.5]*nd,
                                      length_scale_bounds=(0.15, 3.0), nu=2.5)
    gp = GaussianProcessRegressor(kernel=k, n_restarts_optimizer=15,
                                   alpha=alpha, normalize_y=False)
    gp.fit(X_n, y_c)
    return gp, y_c, y_mean

def predict_strw(gp, y_mean, bounds, X_raw):
    mu, sig = gp.predict(normalize(X_raw, bounds), return_std=True)
    return np.exp(mu + y_mean), sig

def recommend(gp, y_c, y_mean, bounds, transform_fn, b_lo=1.0,
              acq='ei', kappa=2.0, n_cand=20000, seed=0):
    np.random.seed(seed)
    b_c = np.random.uniform(b_lo, 8.0, n_cand)
    H_c = np.random.uniform(12.0, 23.0, n_cand)
    X_c = transform_fn(b_c, H_c)
    mu, sig = gp.predict(normalize(X_c, bounds), return_std=True)
    if acq == 'ei':
        best = y_c.max()
        Z = np.where(sig > 1e-9, (mu - best)/sig, 0.0)
        score = np.where(sig > 1e-9, (mu-best)*norm.cdf(Z) + sig*norm.pdf(Z), 0.0)
    else:  # ucb
        score = mu + kappa*sig
    idx = np.argmax(score)
    return b_c[idx], H_c[idx]

STRONG = 31.75

In [ ]:
# ── load data (GitHub raw, not local paths) ────────────────────────────
df_sub  = load_csv_from_github("lhs16_subset_bH_even_n13.csv")
df_full = load_csv_from_github("I_beam_data_2var.csv")

b_sub  = df_sub['b_web_mm'].values.astype(float)
H_sub  = df_sub['H_web_mm'].values.astype(float)
y_sub  = df_sub['Str/w N/g'].values.astype(float)

b_all  = df_full['b_web_mm'].values.astype(float)
H_all  = df_full['H_web_mm'].values.astype(float)
y_all  = df_full['Str/w N/g'].values.astype(float)

print(f'13-beam subset: {len(df_sub)} rows, Str/w in [{y_sub.min():.1f}, {y_sub.max():.1f}]')
print(f'Full dataset:   {len(df_full)} rows, Str/w in [{y_all.min():.1f}, {y_all.max():.1f}]')
print(f'Strong-zone beams (>={STRONG}): {(y_all >= STRONG).sum()}')

In [ ]:
# ── ground-truth GP (b_dH_Pltb_Pbend, all beams, alpha=3e-5) ──────────
X_gt = to_bdHPP(b_all, H_all)
bds_gt = _bounds_from_data(X_gt, ['b','dH','Pltb_m','Pbend_m'])
bds_gt['b'] = (1.0, 8.0)
gp_gt, _, ym_gt = train_gp(X_gt, y_all, bds_gt, alpha=3e-5)
print('GT GP trained.')
print('GT kernel:', gp_gt.kernel_)

In [ ]:
# ── student GPs on 13-beam subset ──────────────────────────────────────
ALPHA = 1e-4

bds_bH = {'b': (1.0, 8.0), 'H': (12.0, 23.0)}
gp_bH, yc_bH, ym_bH = train_gp(to_bH(b_sub, H_sub), y_sub, bds_bH, alpha=ALPHA)

X_PP_tr = to_PP(b_sub, H_sub)
bds_PP  = _bounds_from_data(X_PP_tr, ['Pltb_m', 'Pbend_m'])
gp_PP, yc_PP, ym_PP = train_gp(X_PP_tr, y_sub, bds_PP, alpha=ALPHA)

print('b,H kernel:  ', gp_bH.kernel_)
print('PP kernel:   ', gp_PP.kernel_)

In [ ]:
# ── acquire recommendations ────────────────────────────────────────────
acq_configs = [
    ('EI',     'ei',  None),
    ('UCB κ=1','ucb', 1.0),
    ('UCB κ=2','ucb', 2.0),
    ('UCB κ=3','ucb', 3.0),
]

recs = {}
for label, atype, kap in acq_configs:
    kw = dict(kappa=kap) if kap else {}
    b_bH, H_bH = recommend(gp_bH, yc_bH, ym_bH, bds_bH, to_bH,  acq=atype, **kw)
    b_PP, H_PP = recommend(gp_PP, yc_PP, ym_PP, bds_PP, to_PP,   acq=atype, **kw)

    # GT evaluation at each recommendation
    gt_bH, _ = predict_strw(gp_gt, ym_gt, bds_gt, to_bdHPP(np.array([b_bH]), np.array([H_bH])))
    gt_PP, _ = predict_strw(gp_gt, ym_gt, bds_gt, to_bdHPP(np.array([b_PP]), np.array([H_PP])))

    recs[label] = {
        'bH_rec':   (b_bH, H_bH),
        'PP_rec':   (b_PP, H_PP),
        'bH_gt':    float(gt_bH[0]),
        'PP_gt':    float(gt_PP[0]),
        'PP_in_bH': (_Pltb_m(b_bH, H_bH), _Pbend_m(b_bH, H_bH)),
        'PP_in_PP': (_Pltb_m(b_PP, H_PP), _Pbend_m(b_PP, H_PP)),
    }
    print(f"{label:10s}  b,H: ({b_bH:.2f}, {H_bH:.2f}) GT={gt_bH[0]:.1f}   "
          f"PP: ({b_PP:.2f}, {H_PP:.2f}) GT={gt_PP[0]:.1f}")

In [ ]:
# ── build background grids ─────────────────────────────────────────────
RES = 80

# (b, H) grid
b_vec = np.linspace(1, 8, RES)
H_vec = np.linspace(12, 23, RES)
BG, HG = np.meshgrid(b_vec, H_vec)

b_flat, H_flat = BG.ravel(), HG.ravel()
X_bg = to_bdHPP(b_flat, H_flat)
gt_bg, _ = predict_strw(gp_gt, ym_gt, bds_gt, X_bg)
GT_BH = gt_bg.reshape(RES, RES)

# (Pltb_m, Pbend_m) grid via scatter-interpolation from the (b,H) grid
Pl_all_pts = np.array([_Pltb_m(b, H) for b, H in zip(b_flat, H_flat)])
Pb_all_pts = np.array([_Pbend_m(b, H) for b, H in zip(b_flat, H_flat)])
Pl_obs = np.array([_Pltb_m(b, H) for b, H in zip(b_all, H_all)])
Pb_obs = np.array([_Pbend_m(b, H) for b, H in zip(b_all, H_all)])

pl_lo, pl_hi = Pl_obs.min()*0.85, Pl_obs.max()*1.1
pb_lo, pb_hi = Pb_obs.min()*0.85, Pb_obs.max()*1.1
Pl_vec = np.linspace(pl_lo, pl_hi, RES)
Pb_vec = np.linspace(pb_lo, pb_hi, RES)
PLG, PBG = np.meshgrid(Pl_vec, Pb_vec)

pts_src = np.column_stack([Pl_all_pts, Pb_all_pts])
pts_dst = np.column_stack([PLG.ravel(), PBG.ravel()])
GT_PP = griddata(pts_src, gt_bg, pts_dst, method='linear').reshape(RES, RES)

print('Background grids computed.')

In [ ]:
# ── plot ───────────────────────────────────────────────────────────────
# Short labels for annotations (avoid legend clutter)
short = ['EI', 'U1', 'U2', 'U3']
colors  = ['#e41a1c', '#ff7f00', '#4daf4a', '#984ea3']
markers = ['o', '^', 's', 'D']
vmin, vmax = 20.0, 33.0
levels = np.linspace(vmin, vmax, 30)

Pl_sub = np.array([_Pltb_m(b, H) for b, H in zip(b_sub, H_sub)])
Pb_sub = np.array([_Pbend_m(b, H) for b, H in zip(b_sub, H_sub)])

def annotate_rec(ax, x, y, lbl, col, dx=0.0, dy=0.25):
    ax.annotate(lbl, xy=(x, y), xytext=(x+dx, y+dy),
                color=col, fontsize=8, fontweight='bold',
                ha='center', va='bottom',
                arrowprops=dict(arrowstyle='-', color=col, lw=0.8))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('GP recommendations — 13-beam training, Matern α=1e-4\n'
             'Background: GT GP (all 34 beams, b_dH_Pltb_Pbend) | '
             'filled=PP-param, open=b,H-param', fontsize=10)

# --- LEFT: (b, H) space ---
ax = axes[0]
cf = ax.contourf(BG, HG, GT_BH, levels=levels, cmap='viridis', vmin=vmin, vmax=vmax)
ax.contour(BG, HG, GT_BH, levels=[STRONG], colors='white', linewidths=1.5, linestyles='--')
ax.text(1.15, 12.4, f'GT={STRONG}', color='white', fontsize=7.5)
plt.colorbar(cf, ax=ax, label='GT Str/w (N/g)')

ax.scatter(b_all, H_all, s=18, facecolors='none', edgecolors='white',
           linewidths=0.6, alpha=0.45, zorder=3)
ax.scatter(b_sub, H_sub, c=y_sub, cmap='viridis', vmin=vmin, vmax=vmax,
           s=80, edgecolors='k', linewidths=1.2, zorder=5)

# b,H-param recs (open markers) — may all overlap; annotate the cluster once
bH_xs = [recs[lbl]['bH_rec'][0] for lbl,_,_ in acq_configs]
bH_ys = [recs[lbl]['bH_rec'][1] for lbl,_,_ in acq_configs]
all_same_bH = len(set(zip([round(x,2) for x in bH_xs],
                           [round(y,2) for y in bH_ys]))) == 1
if all_same_bH:
    ax.scatter(bH_xs[0], bH_ys[0], s=260, facecolors='none', edgecolors='gray',
               linewidths=2, zorder=9)
    ax.annotate('EI/U1/U2/U3\n(b,H param)', xy=(bH_xs[0], bH_ys[0]),
                xytext=(bH_xs[0]+0.6, bH_ys[0]+1.0), fontsize=7.5, color='gray',
                arrowprops=dict(arrowstyle='->', color='gray', lw=0.9))
else:
    for (lbl,_,_), col, mk, sl in zip(acq_configs, colors, markers, short):
        x, y = recs[lbl]['bH_rec']
        ax.scatter(x, y, color=col, marker=mk, s=180, facecolors='none',
                   edgecolors=col, linewidths=2, zorder=9)
        annotate_rec(ax, x, y, sl+'\nbH', col)

# PP-param recs (filled markers)
for (lbl,_,_), col, mk, sl in zip(acq_configs, colors, markers, short):
    x, y = recs[lbl]['PP_rec']
    ax.scatter(x, y, color=col, marker=mk, s=180, edgecolors='k',
               linewidths=1.2, zorder=10)
    annotate_rec(ax, x, y, sl, col, dy=0.5)

ax.set_xlabel('b (mm)'); ax.set_ylabel('H (mm)')
ax.set_title('(b, H) space  |  filled = PP-param rec, open = b,H-param rec')

# --- RIGHT: (Pltb_m, Pbend_m) space ---
ax = axes[1]
cf2 = ax.contourf(PLG, PBG, GT_PP, levels=levels, cmap='viridis', vmin=vmin, vmax=vmax)
ax.contour(PLG, PBG, GT_PP, levels=[STRONG], colors='white', linewidths=1.5, linestyles='--')
plt.colorbar(cf2, ax=ax, label='GT Str/w (N/g)')

ax.scatter(Pl_obs, Pb_obs, s=18, facecolors='none', edgecolors='white',
           linewidths=0.6, alpha=0.45, zorder=3)
ax.scatter(Pl_sub, Pb_sub, c=y_sub, cmap='viridis', vmin=vmin, vmax=vmax,
           s=80, edgecolors='k', linewidths=1.2, zorder=5)

pp_span = PLG.max() - PLG.min()
for (lbl,_,_), col, mk, sl in zip(acq_configs, colors, markers, short):
    pl_bH, pb_bH = recs[lbl]['PP_in_bH']
    pl_PP, pb_PP = recs[lbl]['PP_in_PP']
    ax.scatter(pl_PP, pb_PP, color=col, marker=mk, s=180, edgecolors='k',
               linewidths=1.2, zorder=10)
    annotate_rec(ax, pl_PP, pb_PP, sl, col, dx=pp_span*0.04, dy=pp_span*0.04)
    ax.scatter(pl_bH, pb_bH, color=col, marker=mk, s=90, facecolors='none',
               edgecolors=col, linewidths=2, zorder=9)

ax.set_xlabel('P_ltb / mass (N/g)'); ax.set_ylabel('P_bend / mass (N/g)')
ax.set_title('(Pltb_m, Pbend_m) space  |  filled = PP-param, open = b,H-param')

# shared legend strip
from matplotlib.lines import Line2D
handles = ([Line2D([0],[0], color=c, marker=m, ms=8, ls='none',
                   markerfacecolor=c, label=f'{sl} ({lbl})')
            for (lbl,_,_), c, m, sl in zip(acq_configs, colors, markers, short)] +
           [Line2D([0],[0], color='gray', marker='o', ms=8, ls='none',
                   markerfacecolor='none', label='b,H param (open)'),
            Line2D([0],[0], color='k', marker='o', ms=8, ls='none',
                   markerfacecolor='gray', label='PP param (filled)')])
fig.legend(handles=handles, loc='lower center', ncol=6, fontsize=8,
           bbox_to_anchor=(0.5, -0.04))

plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.savefig('viz_gp_recs_n13.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── text summary ───────────────────────────────────────────────────────
print(f"{'Acq':10s}  {'b,H param rec':>28s}  {'PP param rec':>26s}")
print('-'*70)
for label, _, _ in acq_configs:
    b_bH, H_bH = recs[label]['bH_rec']
    b_PP, H_PP = recs[label]['PP_rec']
    gt_bH = recs[label]['bH_gt']
    gt_PP = recs[label]['PP_gt']
    zone_bH = 'STRONG' if gt_bH >= STRONG else f'{gt_bH:.1f}'
    zone_PP = 'STRONG' if gt_PP >= STRONG else f'{gt_PP:.1f}'
    print(f"{label:10s}  b={b_bH:.2f} H={H_bH:.2f} GT={zone_bH:>6s}  "
          f"b={b_PP:.2f} H={H_PP:.2f} GT={zone_PP:>6s}")